# 中证800 V82 full_v46 生产级模型训练导出器

定位：只负责数据校验、训练计划生成、full_v46 模型训练、pkl 导出和完整性检查，不做策略筛选和回测实验。

默认导出：

- `full_v46 + expanding_min36 + legacy_rebalance`，cutoff: 2023/2024/2025；
- `full_v46 + rolling60m + legacy_rebalance`，cutoff: 2023/2024/2025。

如果要做严格标签边界导出，把 `LABEL_BOUNDARY_MODES` 改成 `['legacy_rebalance', 'label_end_safe']`。

## 0. 导入与进度条

In [ ]:
import os
import gc
import json
import pickle
import shutil
import hashlib
import warnings
import builtins as _bi
from pathlib import Path
from datetime import datetime

import lightgbm as lgb
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 220)
pd.set_option("display.width", 240)
pd.set_option("display.max_rows", 120)

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


def progress_iter(iterable, total=None, desc="progress", leave=True):
    if tqdm is not None:
        return tqdm(iterable, total=total, desc=desc, leave=leave)
    def _gen():
        every = _bi.max(1, int((total or 100) / 20))
        for i, item in enumerate(iterable, 1):
            if i == 1 or i % every == 0 or (total is not None and i == total):
                print("%s %s%s" % (desc, i, "/%s" % total if total else ""))
            yield item
    return _gen()


def display_df(df, n=30):
    try:
        display(df.head(n))
    except Exception:
        print(df.head(n).to_string(index=False))


## 1. 生产配置

In [ ]:
# =========================
# Product config
# =========================
PROJECT_DIR = Path.cwd()
PRODUCT_VERSION = "v82"
RUN_NAME = "full_v46_production_model_trainer"
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

OUT_DIR = PROJECT_DIR / "csi800_ml_v82_full_v46_model_trainer_outputs"
MODEL_EXPORT_DIR = OUT_DIR / "model_exports"
MANIFEST_DIR = OUT_DIR / "manifests"
QA_DIR = OUT_DIR / "qa"
for _p in [OUT_DIR, MODEL_EXPORT_DIR, MANIFEST_DIR, QA_DIR]:
    _p.mkdir(parents=True, exist_ok=True)

DATA_CANDIDATES = [
    Path("train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"),
    Path("train_csi800_factor_v40_data_enhancement.csv"),
    Path("data/train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"),
    Path("data/train_csi800_factor_v40_data_enhancement.csv"),
    PROJECT_DIR / "train_csi800_factor_v40_data_enhancement_20190101_20260531.csv",
    PROJECT_DIR / "train_csi800_factor_v40_data_enhancement.csv",
    PROJECT_DIR / "data" / "train_csi800_factor_v40_data_enhancement_20190101_20260531.csv",
    PROJECT_DIR / "data" / "train_csi800_factor_v40_data_enhancement.csv",
]
DATA_PATH_OVERRIDE = None

TARGET_COL = "alpha_1m"
STOCK_COL = "stock"
DATE_COL = "rebalance_date"
FACTOR_DATE_COL = "feature_date"
INDUSTRY_COL = "industry_bucket"
BENCHMARK = "000906.XSHG"

MODEL_FAMILY = "full_v46"
MODEL_ROLE = "production_candidate"
MODEL_VERSION_PREFIX = "v82"

FIXED_ITER = 120
SEED = 42
CORR_THRESHOLD = 0.70
MIN_TRAIN_MONTHS = 36
PICKLE_PROTOCOL = 2

TOP_N_CANDIDATES = 30
STOCK_NUM = 8
PORTFOLIO_RULE = "top8_board_cap"
BOARD_CAPS = {"chinext": 3, "star": 2}
BOARD_CAPS_TEXT = ";".join(["%s:%s" % (k, BOARD_CAPS[k]) for k in sorted(BOARD_CAPS)])
INDUSTRY_CAP_RATIO = 0.20

# 默认只导出 V46 对齐口径；严格生产审计可加上 label_end_safe。
LABEL_BOUNDARY_MODES = ["legacy_rebalance"]
# LABEL_BOUNDARY_MODES = ["legacy_rebalance", "label_end_safe"]

TRAIN_CUTOFFS = ["2023-12-31", "2024-12-31", "2025-12-31"]
TRAIN_POLICIES = [
    {"train_policy": "expanding_min36", "method_type": "expanding", "train_window_months": None, "min_train_months": 36, "description": "expand from 2019-01"},
    {"train_policy": "rolling60m", "method_type": "rolling", "train_window_months": 60, "min_train_months": 60, "description": "latest 60 rebalance months"},
]
RUN_TRAIN_POLICIES = None  # example: ["expanding_min36"]
RUN_CUTOFFS = None         # example: ["2025-12-31"]

EXPORT_MODELS = True
RELOAD_CHECK = True
FAIL_ON_MISSING_FULL_FEATURES = True
SMOKE_TEST = False
SMOKE_MAX_TASKS = 1

BASE_PARAMS_FF10 = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 200,
    "feature_fraction": 1.0,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 0.3,
    "verbose": -1,
}

if RUN_TRAIN_POLICIES is not None:
    _allow = set(RUN_TRAIN_POLICIES)
    TRAIN_POLICIES = [x for x in TRAIN_POLICIES if x["train_policy"] in _allow]
if RUN_CUTOFFS is not None:
    _cut = set([str(x) for x in RUN_CUTOFFS])
    TRAIN_CUTOFFS = [x for x in TRAIN_CUTOFFS if str(x) in _cut]

print("OUT_DIR:", OUT_DIR)
print("MODEL_EXPORT_DIR:", MODEL_EXPORT_DIR)
print("LABEL_BOUNDARY_MODES:", LABEL_BOUNDARY_MODES)
print("TRAIN_POLICIES:", [x["train_policy"] for x in TRAIN_POLICIES])
print("TRAIN_CUTOFFS:", TRAIN_CUTOFFS)
print("fixed_iter:", FIXED_ITER, "seed:", SEED)


## 2. full_v46 特征定义与 LGB 参数

In [ ]:
def unique_keep_order(cols):
    seen = set()
    out = []
    for col in cols:
        if col not in seen:
            out.append(col)
            seen.add(col)
    return out


BASE_FACTOR_COLS = [
    "cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield", "sales_to_price_ratio",
    "cash_earnings_to_price_ratio", "earnings_to_price_ratio", "roe_ttm", "roa_ttm",
    "gross_profit_ttm", "operating_profit_to_total_profit", "net_operate_cash_flow_to_total_liability",
    "net_operating_cash_flow_coverage", "adjusted_profit_to_total_profit", "ACCA", "growth",
    "net_working_capital", "operating_profit_per_share", "net_operate_cash_flow_per_share",
    "total_operating_revenue_per_share", "super_quick_ratio", "MLEV", "debt_to_equity_ratio",
    "debt_to_tangible_equity_ratio", "momentum", "Rank1M", "sharpe_ratio_60", "Variance20",
    "liquidity", "beta", "ATR6", "MFI14", "DAVOL10", "VOL10", "VMACD", "VOSC",
    "Skewness20", "Kurtosis20",
]
HYBRID_LIGHT_EXTRA_COLS = [
    "liq_money_ratio_20_60", "liq_paused_count_20", "px_close_to_ma60", "px_drawdown_60",
    "ts_cash_flow_to_price_ratio_rank_mean_3m", "ts_Rank1M_rank_chg_1m",
]
FULL_V46_COLS = unique_keep_order(BASE_FACTOR_COLS + HYBRID_LIGHT_EXTRA_COLS)

FEATURE_FAMILIES = [
    {
        "family": "full_v46",
        "feature_variant": "full_v46",
        "description": "V46/V61 full hybrid-light feature stack",
        "candidate_cols": FULL_V46_COLS,
    },
]

feature_manifest_rows = []
for fam in FEATURE_FAMILIES:
    feature_manifest_rows.append({
        "family": fam["family"],
        "feature_variant": fam["feature_variant"],
        "candidate_feature_count": len(fam["candidate_cols"]),
        "description": fam["description"],
        "candidate_features": ",".join(fam["candidate_cols"]),
    })
feature_manifest_df = pd.DataFrame(feature_manifest_rows)
feature_manifest_df.to_csv(MANIFEST_DIR / "v82_feature_manifest.csv", index=False)
display_df(feature_manifest_df)


## 3. 通用工具函数

In [ ]:
def resolve_data_path():
    if DATA_PATH_OVERRIDE:
        p = Path(DATA_PATH_OVERRIDE)
        if p.exists():
            return p
        raise IOError("DATA_PATH_OVERRIDE not found: %s" % p)
    candidates = [Path(x) for x in DATA_CANDIDATES]
    for p in candidates:
        if p.exists():
            return p
    searched = [str(p.resolve()) for p in candidates]
    raise IOError("training data csv not found. Put train_csi800_factor_v40_data_enhancement*.csv in one of: %s" % searched)


def safe_to_datetime(df, cols):
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col], errors="coerce").dt.normalize()
    return out


def safe_rank_ic(a, b):
    s = pd.DataFrame({"a": np.asarray(a, dtype=float), "b": np.asarray(b, dtype=float)})
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3 or s["a"].nunique() < 2 or s["b"].nunique() < 2:
        return np.nan
    return s["a"].rank(pct=True).corr(s["b"].rank(pct=True))


def stable_hash_text(text):
    return hashlib.md5(str(text).encode("utf-8")).hexdigest()


def month_ordinal(ts):
    t = pd.Timestamp(ts)
    return int(t.year * 12 + t.month)


def label_mode_tag(mode):
    if mode == "legacy_rebalance":
        return "legacy"
    if mode == "label_end_safe":
        return "safe"
    return str(mode).replace(" ", "_")


def train_start_for_policy(policy, cutoff):
    cutoff = pd.Timestamp(cutoff).normalize()
    if policy.get("train_window_months") is None:
        return pd.Timestamp("2019-01-01")
    months = int(policy["train_window_months"])
    return (cutoff - pd.DateOffset(months=months - 1)).replace(day=1)


def normalize_task(task):
    out = dict(task)
    out["train_start"] = pd.Timestamp(out["train_start"]).normalize()
    out["train_end"] = pd.Timestamp(out["train_end"]).normalize()
    out["test_start"] = (out["train_end"] + pd.DateOffset(months=1)).replace(day=1)
    return out


def build_corr_components(train_df, feature_cols, threshold):
    from collections import defaultdict
    corr = train_df[feature_cols].corr()
    graph = defaultdict(list)
    for i in range(len(feature_cols)):
        for j in range(i + 1, len(feature_cols)):
            v = corr.iloc[i, j]
            if not pd.isnull(v) and abs(v) > threshold:
                graph[feature_cols[i]].append(feature_cols[j])
                graph[feature_cols[j]].append(feature_cols[i])
    for col in feature_cols:
        graph[col]
    visited = set()
    comps = []

    def dfs(x, comp):
        visited.add(x)
        comp.append(x)
        for y in graph[x]:
            if y not in visited:
                dfs(y, comp)

    for col in feature_cols:
        if col not in visited:
            comp = []
            dfs(col, comp)
            comps.append(comp)
    return comps


def select_features_train_only(train_df, candidate_cols):
    cols = unique_keep_order([c for c in candidate_cols if c in train_df.columns])
    if len(cols) == 0:
        raise ValueError("no candidate feature exists in train data")
    missing = train_df[cols].isnull().sum().to_dict()
    keep = []
    remove = []
    for comp in build_corr_components(train_df, cols, CORR_THRESHOLD):
        if len(comp) == 1:
            keep.append(comp[0])
        else:
            comp = _bi.sorted(comp, key=lambda x: (missing[x], x))
            keep.append(comp[0])
            remove.extend(comp[1:])
    return keep, remove


def prepare_xy(df, feature_cols, target_col, fill_values=None):
    d = df.dropna(subset=[target_col]).copy()
    X = d.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan).copy()
    y = d[target_col].astype(float).copy()
    if fill_values is None:
        fill_values = X.median().replace([np.inf, -np.inf], np.nan).fillna(0)
    X = X.fillna(fill_values).fillna(0)
    return X, y, fill_values, d.index


def split_diag_valid(train_df):
    months = _bi.sorted(pd.to_datetime(train_df[DATE_COL].dropna().unique()))
    if len(months) <= 8:
        return train_df.copy(), train_df.copy()
    n_valid = _bi.max(6, int(round(len(months) * 0.20)))
    n_valid = _bi.min(n_valid, len(months) - 1)
    valid_months = set(months[-n_valid:])
    fit = train_df[~train_df[DATE_COL].isin(valid_months)].copy()
    valid = train_df[train_df[DATE_COL].isin(valid_months)].copy()
    if len(fit) == 0 or len(valid) == 0:
        return train_df.copy(), train_df.copy()
    return fit, valid


def make_train_df(df_all, task):
    task = normalize_task(task)
    mode = task["label_boundary_mode"]
    mask = (df_all[DATE_COL] >= task["train_start"]) & (df_all[DATE_COL] <= task["train_end"])
    if mode == "label_end_safe":
        mask = mask & (df_all["next_date"] <= task["train_end"])
    elif mode != "legacy_rebalance":
        raise ValueError("unknown label boundary mode: " + str(mode))
    return df_all[mask].copy()


def train_direct_lgb(train_df, feature_cols):
    params = dict(BASE_PARAMS_FF10)
    params["seed"] = SEED
    X_train, y_train, fill_values, _ = prepare_xy(train_df, feature_cols, TARGET_COL)
    if len(X_train) == 0:
        raise ValueError("empty training matrix")
    model = lgb.train(params, lgb.Dataset(X_train, label=y_train), num_boost_round=_bi.max(1, int(FIXED_ITER)))
    pred = np.asarray(model.predict(X_train[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)
    return {"model": model, "fill_values": fill_values, "train_rows": int(len(X_train)), "train_rank_ic": safe_rank_ic(y_train, pred)}


def needs_v4_adapter(feature_cols):
    adapter_cols = set(HYBRID_LIGHT_EXTRA_COLS)
    return any(c in adapter_cols for c in feature_cols)


## 4. 数据模块：加载、校验、压缩

In [ ]:
def load_dataset(path):
    df = pd.read_csv(path)
    df = safe_to_datetime(df, [DATE_COL, FACTOR_DATE_COL, "feature_date", "next_date"])
    if STOCK_COL not in df.columns:
        for alt in ["code", "security", "order_book_id"]:
            if alt in df.columns:
                df = df.rename(columns={alt: STOCK_COL})
                break
    if FACTOR_DATE_COL not in df.columns and "feature_date" in df.columns:
        df[FACTOR_DATE_COL] = df["feature_date"]
    if FACTOR_DATE_COL not in df.columns:
        df[FACTOR_DATE_COL] = df[DATE_COL]
    if "next_date" not in df.columns:
        df["next_date"] = df[DATE_COL]
    if TARGET_COL not in df.columns:
        if "raw_return_1m" in df.columns and "benchmark_csi800_1m" in df.columns:
            df[TARGET_COL] = pd.to_numeric(df["raw_return_1m"], errors="coerce") - pd.to_numeric(df["benchmark_csi800_1m"], errors="coerce")
        else:
            raise ValueError("target column not found: " + TARGET_COL)
    if INDUSTRY_COL not in df.columns:
        df[INDUSTRY_COL] = "UNKNOWN"
    need = [STOCK_COL, DATE_COL, TARGET_COL, INDUSTRY_COL, FACTOR_DATE_COL, "next_date"]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise ValueError("dataset missing columns: " + ",".join(missing))
    df[STOCK_COL] = df[STOCK_COL].astype(str)
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    df = df.dropna(subset=[STOCK_COL, DATE_COL, TARGET_COL]).copy()
    df = df.sort_values([DATE_COL, STOCK_COL]).reset_index(drop=True)
    return df


def audit_dataset(df):
    rows = []
    rows.append({"check": "rows", "value": int(len(df)), "status": "ok" if len(df) > 0 else "fail", "detail": ""})
    rows.append({"check": "date_min", "value": str(df[DATE_COL].min()), "status": "info", "detail": ""})
    rows.append({"check": "date_max", "value": str(df[DATE_COL].max()), "status": "info", "detail": ""})
    rows.append({"check": "months", "value": int(df[DATE_COL].nunique()), "status": "ok" if df[DATE_COL].nunique() >= MIN_TRAIN_MONTHS else "fail", "detail": ""})
    dup_count = int(df.duplicated([STOCK_COL, DATE_COL]).sum())
    rows.append({"check": "duplicate_stock_date", "value": dup_count, "status": "ok" if dup_count == 0 else "warn", "detail": "duplicates are kept last by downstream only if you fix input"})
    target_nan = int(df[TARGET_COL].isnull().sum())
    rows.append({"check": "target_nan", "value": target_nan, "status": "ok" if target_nan == 0 else "warn", "detail": ""})
    feature_missing = [c for c in FULL_V46_COLS if c not in df.columns]
    rows.append({"check": "full_v46_missing_features", "value": len(feature_missing), "status": "ok" if len(feature_missing) == 0 else "fail", "detail": ",".join(feature_missing)})
    audit_df = pd.DataFrame(rows)
    return audit_df


DATA_PATH = resolve_data_path()
df_all = load_dataset(DATA_PATH)
data_audit_df = audit_dataset(df_all)
data_audit_df.to_csv(QA_DIR / "v82_data_audit.csv", index=False)

print("DATA_PATH:", DATA_PATH)
print("loaded:", df_all.shape)
print("rebalance_date:", df_all[DATE_COL].min(), "->", df_all[DATE_COL].max())
print("next_date:", df_all["next_date"].min(), "->", df_all["next_date"].max())
display_df(data_audit_df, 40)
display_df(df_all[[TARGET_COL]].describe().T, 5)

fail_checks = data_audit_df[data_audit_df["status"] == "fail"]
if len(fail_checks):
    display_df(fail_checks, 20)
    raise ValueError("data audit failed; fix input data before training")
if FAIL_ON_MISSING_FULL_FEATURES:
    missing = [c for c in FULL_V46_COLS if c not in df_all.columns]
    if missing:
        raise ValueError("missing full_v46 features: " + ",".join(missing))

# Keep only production training columns to reduce memory.
keep_cols = unique_keep_order([STOCK_COL, DATE_COL, TARGET_COL, INDUSTRY_COL, FACTOR_DATE_COL, "next_date", "raw_return_1m", "benchmark_csi800_1m"] + FULL_V46_COLS)
keep_cols = [c for c in keep_cols if c in df_all.columns]
df_all = df_all[keep_cols].copy()
for col in progress_iter(FULL_V46_COLS + [TARGET_COL], total=len(FULL_V46_COLS) + 1, desc="compact float32"):
    if col in df_all.columns and df_all[col].dtype == np.float64:
        df_all[col] = df_all[col].astype(np.float32)
gc.collect()
print("compacted:", df_all.shape)


## 5. 训练计划生成

In [ ]:
def build_training_tasks(df):
    data_min = pd.Timestamp(df[DATE_COL].min()).normalize()
    data_max = pd.Timestamp(df[DATE_COL].max()).normalize()
    rows = []
    for mode in LABEL_BOUNDARY_MODES:
        for policy in TRAIN_POLICIES:
            for cutoff_str in TRAIN_CUTOFFS:
                cutoff = pd.Timestamp(cutoff_str).normalize()
                train_start = train_start_for_policy(policy, cutoff)
                if train_start < data_min:
                    train_start = data_min
                train_mask = (df[DATE_COL] >= train_start) & (df[DATE_COL] <= cutoff)
                if mode == "label_end_safe":
                    train_mask = train_mask & (df["next_date"] <= cutoff)
                elif mode != "legacy_rebalance":
                    raise ValueError("unknown label boundary mode: " + str(mode))
                train_months = int(df.loc[train_mask, DATE_COL].nunique())
                train_rows = int(train_mask.sum())
                min_months = int(policy.get("min_train_months", MIN_TRAIN_MONTHS))
                if train_months < min_months:
                    print("skip too few train months", mode, policy["train_policy"], cutoff, train_months)
                    continue
                tag = "%s__%s__cutoff%s" % (label_mode_tag(mode), policy["train_policy"], cutoff.strftime("%Y%m%d"))
                rows.append({
                    "task_id": "%s__full_v46" % tag,
                    "family": "full_v46",
                    "feature_variant": "full_v46",
                    "label_boundary_mode": mode,
                    "train_policy": policy["train_policy"],
                    "method_type": policy["method_type"],
                    "train_window_months": policy.get("train_window_months"),
                    "train_start": train_start,
                    "train_end": cutoff,
                    "test_start": (cutoff + pd.DateOffset(months=1)).replace(day=1),
                    "train_months": train_months,
                    "train_rows_expected": train_rows,
                    "policy_description": policy.get("description", ""),
                    "data_min_date": data_min,
                    "data_max_date": data_max,
                })
    task_df = pd.DataFrame(rows)
    if len(task_df):
        task_df = task_df.sort_values(["label_boundary_mode", "train_policy", "train_end"]).reset_index(drop=True)
    if SMOKE_TEST and len(task_df):
        task_df = task_df.head(SMOKE_MAX_TASKS).copy()
    return task_df


training_task_df = build_training_tasks(df_all)
training_task_df.to_csv(MANIFEST_DIR / "v82_training_task_plan.csv", index=False)
print("training tasks:", training_task_df.shape)
display_df(training_task_df, 40)
if len(training_task_df) == 0:
    raise ValueError("empty training task plan")


## 6. 模型训练与 pkl 导出

In [ ]:
def make_model_id(task, feature_variant):
    return "%s_%s_%s_%s_%s_start%s_cutoff%s_fixed%s" % (
        MODEL_VERSION_PREFIX,
        task["family"],
        feature_variant,
        task["label_boundary_mode"],
        task["train_policy"],
        pd.Timestamp(task["train_start"]).strftime("%Y%m%d"),
        pd.Timestamp(task["train_end"]).strftime("%Y%m%d"),
        int(FIXED_ITER),
    )


def bundle_filename(model_id):
    return "model_%s.pkl" % model_id


def compute_diag_ic(model, diag_valid_df, feature_cols, fill_values):
    X_valid, y_valid, _, _ = prepare_xy(diag_valid_df, feature_cols, TARGET_COL, fill_values)
    if len(X_valid) == 0:
        return np.nan
    pred = np.asarray(model.predict(X_valid[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)
    return safe_rank_ic(y_valid, pred)


def export_model_bundle(task, trained, feature_cols, removed_cols, diag_rank_ic, train_df, variant):
    model_id = make_model_id(task, variant["feature_variant"])
    model_file = bundle_filename(model_id)
    model_path = MODEL_EXPORT_DIR / model_file
    feature_signature = stable_hash_text("|".join(feature_cols))
    data_signature = stable_hash_text("%s|%s|%s|%s|%s" % (str(DATA_PATH), len(df_all), df_all[DATE_COL].min(), df_all[DATE_COL].max(), TARGET_COL))
    fill_values = trained["fill_values"]
    if hasattr(fill_values, "to_dict"):
        fill_values = fill_values.to_dict()
    bundle = {
        "objective": "v210_refit_fixed_iter_overlay",
        "research_version": model_id,
        "product_version": PRODUCT_VERSION,
        "run_name": RUN_NAME,
        "run_timestamp": RUN_TIMESTAMP,
        "benchmark": BENCHMARK,
        "model_family": MODEL_FAMILY,
        "family": task["family"],
        "feature_variant": variant["feature_variant"],
        "model_role": MODEL_ROLE,
        "label_boundary_mode": task["label_boundary_mode"],
        "require_label_end_within_train": bool(task["label_boundary_mode"] == "label_end_safe"),
        "legacy_unsealed_boundary": bool(task["label_boundary_mode"] == "legacy_rebalance"),
        "train_policy": task["train_policy"],
        "method_type": task["method_type"],
        "train_window_months": task.get("train_window_months"),
        "train_start": pd.Timestamp(task["train_start"]),
        "train_end": pd.Timestamp(task["train_end"]),
        "test_start": pd.Timestamp(task["test_start"]),
        "label_end": pd.Timestamp(task["train_end"]),
        "target_col": TARGET_COL,
        "target_note": "direct LGB regression on alpha_1m, fixed iteration, no early stopping",
        "data_file": str(DATA_PATH),
        "data_signature": data_signature,
        "feature_signature": feature_signature,
        "protocol": "v82_full_v46_production_model_export",
        "param_set": "v46_base_ff10_original",
        "base_params": dict(BASE_PARAMS_FF10),
        "base_model": trained["model"],
        "base_feature_cols": list(feature_cols),
        "base_fill_values": dict(fill_values),
        "base_best_iter": int(FIXED_ITER),
        "model_iter": int(FIXED_ITER),
        "fixed_iter": int(FIXED_ITER),
        "base_inner_metrics": {
            "train_rank_ic": float(trained["train_rank_ic"]) if not pd.isnull(trained["train_rank_ic"]) else np.nan,
            "diag_rank_ic": float(diag_rank_ic) if not pd.isnull(diag_rank_ic) else np.nan,
        },
        "base_removed_features": list(removed_cols),
        "overlay_mode": "direct",
        "overlay_weight": 0.0,
        "residual_model": None,
        "residual_feature_cols": [],
        "residual_fill_values": {},
        "top_n_candidates": TOP_N_CANDIDATES,
        "stock_num": STOCK_NUM,
        "portfolio_rule": PORTFOLIO_RULE,
        "board_caps": dict(BOARD_CAPS),
        "board_caps_text": BOARD_CAPS_TEXT,
        "industry_cap_ratio": INDUSTRY_CAP_RATIO,
        "requires_v4_feature_adapter": bool(needs_v4_adapter(feature_cols)),
        "requires_industry_relative_adapter": False,
        "uses_time_weight": False,
        "uses_sample_weight": False,
        "uses_current_valid_for_training": False,
        "train_row_count": int(len(train_df)),
        "train_month_count": int(train_df[DATE_COL].nunique()),
        "max_train_rebalance_date": str(train_df[DATE_COL].max().date()),
        "max_train_next_date": str(train_df["next_date"].max().date()) if "next_date" in train_df.columns else "",
        "note": "full_v46 production training export; compatible with simplified JQ backtest loader",
    }
    if EXPORT_MODELS:
        with open(model_path, "wb") as f:
            pickle.dump(bundle, f, protocol=PICKLE_PROTOCOL)
    return model_id, model_file, model_path, bundle


def train_one_task(task, variant):
    task = normalize_task(task)
    train_df = make_train_df(df_all, task)
    if len(train_df) == 0:
        raise ValueError("empty train_df for task " + str(task.get("task_id")))
    if train_df[DATE_COL].nunique() < int(task.get("min_train_months", MIN_TRAIN_MONTHS)):
        raise ValueError("too few train months for task " + str(task.get("task_id")))
    diag_fit_df, diag_valid_df = split_diag_valid(train_df)
    feature_cols, removed_cols = select_features_train_only(diag_fit_df, variant["candidate_cols"])
    trained = train_direct_lgb(train_df, feature_cols)
    diag_rank_ic = compute_diag_ic(trained["model"], diag_valid_df, feature_cols, trained["fill_values"])
    model_id, model_file, model_path, bundle = export_model_bundle(task, trained, feature_cols, removed_cols, diag_rank_ic, train_df, variant)
    row = {
        "model_id": model_id,
        "model_file": model_file,
        "model_path": str(model_path),
        "family": task["family"],
        "feature_variant": variant["feature_variant"],
        "label_boundary_mode": task["label_boundary_mode"],
        "train_policy": task["train_policy"],
        "method_type": task["method_type"],
        "train_window_months": task.get("train_window_months"),
        "train_start": task["train_start"],
        "train_end": task["train_end"],
        "test_start": task["test_start"],
        "train_rows": int(len(train_df)),
        "train_months": int(train_df[DATE_COL].nunique()),
        "max_train_rebalance_date": str(train_df[DATE_COL].max().date()),
        "max_train_next_date": str(train_df["next_date"].max().date()) if "next_date" in train_df.columns else "",
        "candidate_feature_count": len(variant["candidate_cols"]),
        "selected_feature_count": len(feature_cols),
        "removed_feature_count": len(removed_cols),
        "train_rank_ic": trained["train_rank_ic"],
        "diag_rank_ic": diag_rank_ic,
        "feature_signature": bundle["feature_signature"],
        "data_signature": bundle["data_signature"],
        "requires_v4_feature_adapter": bundle["requires_v4_feature_adapter"],
        "portfolio_rule": PORTFOLIO_RULE,
        "stock_num": STOCK_NUM,
        "board_caps": BOARD_CAPS_TEXT,
        "feature_cols": ",".join(feature_cols),
        "removed_features": ",".join(removed_cols),
    }
    del trained, train_df, diag_fit_df, diag_valid_df
    gc.collect()
    return row


manifest_rows = []
variant = FEATURE_FAMILIES[0]
for _, task_row in progress_iter(training_task_df.iterrows(), total=len(training_task_df), desc="train/export full_v46 models"):
    task = task_row.to_dict()
    row = train_one_task(task, variant)
    manifest_rows.append(row)
    print("exported", row["model_file"], "rows", row["train_rows"], "features", row["selected_feature_count"], "diag_ic", row["diag_rank_ic"])

model_manifest_df = pd.DataFrame(manifest_rows)
model_manifest_df.to_csv(MANIFEST_DIR / "v82_model_manifest.csv", index=False)
model_manifest_df.to_csv(OUT_DIR / "v82_model_manifest.csv", index=False)
display_df(model_manifest_df, 40)


## 7. 上传清单与完整性检查

In [ ]:
upload_rows = []
for _, r in model_manifest_df.iterrows():
    upload_rows.append({
        "model_id": r["model_id"],
        "model_file": r["model_file"],
        "local_model_path": r["model_path"],
        "joinquant_file_hint": "v4模型/csi800_ml_v82_full_v46_model_trainer_outputs/model_exports/" + r["model_file"],
        "family": r["family"],
        "label_boundary_mode": r["label_boundary_mode"],
        "train_policy": r["train_policy"],
        "train_start": r["train_start"],
        "train_end": r["train_end"],
        "selected_feature_count": r["selected_feature_count"],
        "train_rank_ic": r["train_rank_ic"],
        "diag_rank_ic": r["diag_rank_ic"],
        "portfolio_rule": r["portfolio_rule"],
        "stock_num": r["stock_num"],
        "board_caps": r["board_caps"],
    })
upload_list_df = pd.DataFrame(upload_rows)
upload_list_df.to_csv(MANIFEST_DIR / "v82_joinquant_upload_list.csv", index=False)
upload_list_df.to_csv(OUT_DIR / "v82_joinquant_upload_list.csv", index=False)

required_bundle_keys = [
    "objective", "product_version", "base_model", "base_feature_cols", "base_fill_values",
    "overlay_mode", "family", "train_policy", "label_boundary_mode", "top_n_candidates",
    "stock_num", "portfolio_rule", "requires_v4_feature_adapter",
]
check_rows = []
if RELOAD_CHECK:
    for _, r in progress_iter(model_manifest_df.iterrows(), total=len(model_manifest_df), desc="reload exported pkl"):
        path = Path(r["model_path"])
        row = {"model_file": r["model_file"], "path": str(path), "exists": path.exists(), "ok": False, "error": ""}
        try:
            with open(path, "rb") as f:
                bundle = pickle.load(f)
            missing = [k for k in required_bundle_keys if k not in bundle]
            row["missing_keys"] = ",".join(missing)
            row["ok"] = len(missing) == 0
            row["feature_count"] = len(bundle.get("base_feature_cols", []))
            row["overlay_mode"] = bundle.get("overlay_mode")
            row["family"] = bundle.get("family")
            row["train_policy"] = bundle.get("train_policy")
            row["label_boundary_mode"] = bundle.get("label_boundary_mode")
            row["requires_v4_feature_adapter"] = bundle.get("requires_v4_feature_adapter")
        except Exception as err:
            row["error"] = str(err)
        check_rows.append(row)
reload_check_df = pd.DataFrame(check_rows)
reload_check_df.to_csv(QA_DIR / "v82_reload_check.csv", index=False)

if len(reload_check_df) and not bool(reload_check_df["ok"].all()):
    display_df(reload_check_df, 40)
    raise ValueError("reload check failed")

print("saved outputs:")
for path in [
    OUT_DIR / "v82_model_manifest.csv",
    OUT_DIR / "v82_joinquant_upload_list.csv",
    QA_DIR / "v82_data_audit.csv",
    QA_DIR / "v82_reload_check.csv",
]:
    print("-", path)
print("model files:")
for _, r in model_manifest_df.iterrows():
    print("-", r["model_file"])

display_df(upload_list_df, 40)
display_df(reload_check_df, 40)


## 8. 输出说明

In [ ]:
readme_lines = []
readme_lines.append("# V82 full_v46 production model trainer outputs")
readme_lines.append("")
readme_lines.append("Run timestamp: %s" % RUN_TIMESTAMP)
readme_lines.append("Data path: %s" % DATA_PATH)
readme_lines.append("Label boundary modes: %s" % ",".join(LABEL_BOUNDARY_MODES))
readme_lines.append("Train policies: %s" % ",".join([x["train_policy"] for x in TRAIN_POLICIES]))
readme_lines.append("")
readme_lines.append("## Files")
readme_lines.append("- model_exports/*.pkl: upload these pkl files to JoinQuant and set PRIMARY_MODEL_FILE in the backtest.")
readme_lines.append("- v82_model_manifest.csv: training metadata and diagnostics.")
readme_lines.append("- v82_joinquant_upload_list.csv: concise upload checklist.")
readme_lines.append("- qa/v82_data_audit.csv: input data checks.")
readme_lines.append("- qa/v82_reload_check.csv: pkl reload checks.")
readme_lines.append("")
readme_lines.append("## Default recommendation")
readme_lines.append("Use full_v46 + expanding_min36 + legacy_rebalance as mainline; use rolling60m as challenger/shadow.")

readme_path = OUT_DIR / "README_v82_model_trainer.md"
with open(readme_path, "w") as f:
    f.write("\n".join(readme_lines))
print("README:", readme_path)
print("done")
